# Ноутбук с препроцессингом, обучением и сравнением моделей

## 1. Импорт бибилиотек и конфигурация проекта

In [55]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
import pyarrow
import phik
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import mlflow
import os
from datetime import datetime
import category_encoders as ce

In [24]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [25]:
train = pd.read_parquet("../data/features/train_features.parquet")
test = pd.read_parquet("../data/features/test_features.parquet")

In [26]:
y_train = train[CONFIG["TARGET"]]
X_train = train.drop(columns=[CONFIG["TARGET"]])

y_test = test[CONFIG["TARGET"]]
X_test = test.drop(columns=[CONFIG["TARGET"]])

In [59]:
dir = 'C:/mlflow_runs'
os.makedirs(dir, exist_ok=True)
mlflow.set_tracking_uri(f'sqlite:///{dir}/mlflow.db')
mlflow.set_experiment('test_my_first_exp')

<Experiment: artifact_location='file:c:/Users/Степан/Desktop/car-price-analyzer/notebooks/mlruns/1', creation_time=1784637692059, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784637692059, lifecycle_stage='active', name='test_my_first_exp', tags={}, trace_location=None, workspace='default'>

## 2. Пайплайн предобработки

In [27]:
num_cols = X_train.select_dtypes('number').columns.to_list()
low_card_cols = ['Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Цвет', 'Тип кузова']
high_card_cols = ['Марка', 'Регион', 'Модель']

Переводим колонки тпиа object в category для моделей.

In [32]:
cat_cols = low_card_cols + high_card_cols
X_train[cat_cols] = X_train[cat_cols].astype('category')
X_test[cat_cols] = X_test[cat_cols].astype('category')

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', RobustScaler(), num_cols),
        ('low_card', OneHotEncoder(handle_unknown='ignore', sparse_output=False), low_card_cols),
        ('high_card', CatBoostEncoder(handle_unknown='value'), high_card_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge())
])

In [43]:
y_train_orig = np.expm1(y_train)

In [39]:
tt_pipeline = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

## 3. Выбор метрик и базовая модель

In [57]:
scoring_metrics = {
    'mae': 'neg_mean_absolute_error',
    'mape': 'neg_mean_absolute_percentage_error',
    'rmse': 'neg_root_mean_squared_error'
}

In [58]:
run_name = 'baseline_ridge'
with mlflow.start_run(run_name=run_name):
    print('Запуск кросс-валидации...')
    cv_results = cross_validate(
        tt_pipeline,
        X_train,
        y_train_orig,
        cv=5,
        scoring=scoring_metrics,
        n_jobs=-1,
        return_train_score=False
    )

    mean_mae = -cv_results['test_mae'].mean()
    mean_mape = -cv_results['test_mape'].mean()
    mean_rmse = -cv_results['test_rmse'].mean()
    
    print("\nМетрики (Baseline):")
    print(f"MAE:  {mean_mae:.2f}")
    print(f"MAPE: {mean_mape:.2f}")
    print(f"RMSE: {mean_rmse:.2f}")

    mlflow.log_metric('cv_mean_mae', mean_mae)
    mlflow.log_metric('cv_mean_mape', mean_mape)
    mlflow.log_metric('cv_mean_rmse', mean_rmse)

    mlflow.sklearn.log_model(
        sk_model=tt_pipeline,
        name='model',
        serialization_format="pickle"
    )

    print("Эксперимент успешно сохранен в MLflow")

Запуск кросс-валидации...

Метрики (Baseline):
MAE:  236526.19
MAPE: 0.25
RMSE: 769235.95


2026/07/21 16:04:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Эксперимент успешно сохранен в MLflow


## 4. Обучение моделей-кандидатов на кросс-валидации